In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# TVP-VAR (Kalman Filter 기반, 수정 버전)
# ------------------------------------------------------------
# 입력: transformed / scaled data
# lag : BIC 선택
# 결과:
#   - time-varying beta
#   - EWMA residual covariance
#
# 수정 핵심
# 1) cov_t[t] = res @ res.T  제거
# 2) residual covariance는 EWMA 방식으로 추정
# 3) cov_t가 PSD가 되도록 대칭화 + ridge 추가
# 4) 저장되는 cov_t는 GFEVD에 넣을 innovation covariance 역할
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_BETA = BASE_DIR / "tvpvar_beta.npy"
OUT_COV = BASE_DIR / "tvpvar_cov.npy"
OUT_LAG = BASE_DIR / "tvpvar_selected_lag.txt"

# Kalman / covariance 설정
LAMBDA = 0.99          # state persistence
DELTA = 0.01           # measurement noise scale
COV_ALPHA = 0.05       # EWMA covariance smoothing
RIDGE_EPS = 1e-8       # covariance 수치 안정화
INIT_P_SCALE = 0.1     # 초기 state covariance scale
Q_SCALE = 1e-4         # state innovation covariance scale

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)

if "Date" in df.columns:
    df = df.drop(columns=["Date"])

cols = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

df = df[cols].dropna().reset_index(drop=True)

Y = df.values
T, k = Y.shape

print("Data shape:", Y.shape)

# -----------------------------
# Step 1. Lag selection (BIC)
# -----------------------------
var_model = VAR(df)
lag_order = var_model.select_order(maxlags=10)
p = lag_order.selected_orders["bic"]

if p is None or p < 1:
    p = 1

print("Selected lag (BIC):", p)

with open(OUT_LAG, "w", encoding="utf-8") as f:
    f.write(str(p))

# -----------------------------
# Step 2. Prepare matrices
# -----------------------------
def create_lagged_matrix(Y, p):
    T, k = Y.shape
    X = []

    for t in range(p, T):
        row = []
        for i in range(1, p + 1):
            row.extend(Y[t - i])
        X.append(row)

    return np.array(X)

X = create_lagged_matrix(Y, p)     # (T_eff, k*p)
Y_trim = Y[p:]                     # (T_eff, k)

T_eff = Y_trim.shape[0]
k_beta = k * p

# -----------------------------
# Step 3. Kalman Filter
# -----------------------------
beta_t = np.zeros((T_eff, k, k_beta))
cov_t = np.zeros((T_eff, k, k))

# state covariance for one equation
P_t = np.eye(k_beta) * INIT_P_SCALE

# observation covariance (fixed in filter step)
R = np.eye(k) * DELTA

# state innovation covariance
Q = np.eye(k_beta) * Q_SCALE

# initial beta
beta_prev = np.zeros((k, k_beta))

# EWMA innovation covariance initializer
Sigma_ewma = np.cov(Y_trim.T)
Sigma_ewma = np.atleast_2d(Sigma_ewma)
Sigma_ewma = 0.5 * (Sigma_ewma + Sigma_ewma.T)
Sigma_ewma += np.eye(k) * RIDGE_EPS

print("Initial Sigma_ewma diag:", np.round(np.diag(Sigma_ewma), 6))

for t in range(T_eff):
    x_t = X[t]                    # shape: (k_beta,)
    y_t = Y_trim[t]               # shape: (k,)

    beta_new = np.zeros((k, k_beta))
    res_vec = np.zeros(k)

    # --------------------------------------------------
    # equation-by-equation Kalman update
    # y_{i,t} = x_t' beta_{i,t} + eps_{i,t}
    # --------------------------------------------------
    for i in range(k):
        beta_prev_i = beta_prev[i].reshape(-1, 1)    # (k_beta, 1)

        # Prediction
        beta_pred_i = beta_prev_i
        P_pred = P_t / LAMBDA + Q

        # Measurement
        H_i = x_t.reshape(1, -1)                     # (1, k_beta)
        y_pred_i = float(H_i @ beta_pred_i)          # scalar
        resid_i = float(y_t[i] - y_pred_i)

        S_i = float(H_i @ P_pred @ H_i.T + R[i, i])
        if S_i <= 0:
            S_i = RIDGE_EPS

        K_i = (P_pred @ H_i.T) / S_i                 # (k_beta, 1)

        beta_upd_i = beta_pred_i + K_i * resid_i
        P_upd = (np.eye(k_beta) - K_i @ H_i) @ P_pred

        beta_new[i] = beta_upd_i.ravel()
        res_vec[i] = resid_i

        # 다음 식에서 재사용할 P_t는 공통으로 두되,
        # 마지막 업데이트 값을 사용
        P_t = 0.5 * (P_upd + P_upd.T)

    beta_prev = beta_new
    beta_t[t] = beta_new

    # --------------------------------------------------
    # EWMA innovation covariance update
    # 핵심 수정 부분
    # --------------------------------------------------
    outer_res = np.outer(res_vec, res_vec)

    Sigma_ewma = (1.0 - COV_ALPHA) * Sigma_ewma + COV_ALPHA * outer_res
    Sigma_ewma = 0.5 * (Sigma_ewma + Sigma_ewma.T)
    Sigma_ewma += np.eye(k) * RIDGE_EPS

    cov_t[t] = Sigma_ewma

    if t % 200 == 0 or t == T_eff - 1:
        print(
            f"t={t}/{T_eff} | "
            f"mean|beta|={np.mean(np.abs(beta_new)):.6f} | "
            f"mean diag(cov)={np.mean(np.diag(Sigma_ewma)):.6f}"
        )

# -----------------------------
# Save
# -----------------------------
np.save(OUT_BETA, beta_t)
np.save(OUT_COV, cov_t)

print("\nSaved:")
print(" -", OUT_BETA)
print(" -", OUT_COV)
print(" -", OUT_LAG)

# -----------------------------
# Sanity checks
# -----------------------------
print("\n[Sanity Check]")
print("beta_t shape:", beta_t.shape)
print("cov_t shape :", cov_t.shape)
print("last beta mean abs:", np.mean(np.abs(beta_t[-1])))
print("last cov diag     :", np.round(np.diag(cov_t[-1]), 6))
print("last cov eig min  :", np.min(np.linalg.eigvalsh(cov_t[-1])))

Data shape: (1036, 7)
Selected lag (BIC): 1
Initial Sigma_ewma diag: [1.001773 1.001789 0.999179 0.997618 1.000913 1.001582 1.001664]
t=0/1035 | mean|beta|=0.024521 | mean diag(cov)=0.964003
t=200/1035 | mean|beta|=0.195597 | mean diag(cov)=0.786809
t=400/1035 | mean|beta|=0.236593 | mean diag(cov)=2.424330
t=600/1035 | mean|beta|=0.203365 | mean diag(cov)=1.255863
t=800/1035 | mean|beta|=0.167530 | mean diag(cov)=1.113952


C:\Users\User\AppData\Local\Temp\ipykernel_6664\3408585470.py:148: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y_pred_i = float(H_i @ beta_pred_i)          # scalar
C:\Users\User\AppData\Local\Temp\ipykernel_6664\3408585470.py:151: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  S_i = float(H_i @ P_pred @ H_i.T + R[i, i])


t=1000/1035 | mean|beta|=0.252309 | mean diag(cov)=1.681431
t=1034/1035 | mean|beta|=0.211181 | mean diag(cov)=1.920404

Saved:
 - tvpvar_beta.npy
 - tvpvar_cov.npy
 - tvpvar_selected_lag.txt

[Sanity Check]
beta_t shape: (1035, 7, 7)
cov_t shape : (1035, 7, 7)
last beta mean abs: 0.21118143438888548
last cov diag     : [1.869735 3.278184 3.262017 3.731402 0.531051 0.297054 0.473385]
last cov eig min  : 0.19613269836429334
